[Open this notebook in Google Colab](https://colab.research.google.com/github/donleaveher/plasticity-placement/blob/agent%2Fadd-lora-evaluation/notebooks/pathmem_g1_p0/pathmem_g1_p0_colab.ipynb)

# PathMem G1 qualification → gated P0 smoke

This notebook is restart-safe and evidence-preserving. Its safe default only checks the frozen G0 bundle and prints the 24-anchor/44-artifact plan; it does not train or run GPU inference. Use an NVIDIA runtime with native BF16 support (A100, L4, or newer).

## Run passes

Edit **only the next cell** between passes.

1. Inspection: keep all defaults and run all cells.
2. Recipe A diagnosis: keep both run flags false, set `USE_GOOGLE_DRIVE=True` and `RUN_LABEL='pathmem-v1'`; the results cell derives diagnostics without training.
3. Recipe B G1: preserve the failed Recipe A directory, use a new `RUN_LABEL`, set `USE_GOOGLE_DRIVE=True`, `RUN_G1=True`, keep `G1_RECIPE='B'`, and set `APPROVER` to your own name or lab identifier; run all cells.
4. P0: after G1 writes a passing authorization, set `RUN_G1=False`, `RUN_P0=True`, keep the same `APPROVER` and `RUN_LABEL`, then run all cells.

`APPROVER` is a responsible-party identifier, not a password or API key. Never put secrets in this notebook.

In [ ]:
# @title User controls — this is the only cell to edit
RUN_G1 = False  # @param {type:"boolean"}
RUN_P0 = False  # @param {type:"boolean"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
APPROVER = ""  # @param {type:"string"}
RUN_LABEL = "pathmem-g1-recipe-b"  # @param {type:"string"}
G1_RECIPE = "B"  # @param ["A", "B"]


In [ ]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path

if RUN_G1 and RUN_P0:
    raise ValueError("Run G1 and P0 in separate passes")
if (RUN_G1 or RUN_P0) and not APPROVER.strip():
    raise ValueError("APPROVER must identify the responsible human for execution")
if not re.fullmatch(r"[A-Za-z0-9._-]+", RUN_LABEL):
    raise ValueError("RUN_LABEL contains an unsupported character")

if G1_RECIPE not in {"A", "B"}:
    raise ValueError("G1_RECIPE must be A or B")

REPOSITORY_URL = "https://github.com/donleaveher/plasticity-placement.git"
REPOSITORY_BRANCH = "agent/add-lora-evaluation"
REPOSITORY_ROOT = Path("/content/plasticity-placement")
if not REPOSITORY_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
    )
else:
    observed = subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "remote", "get-url", "origin"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if observed != REPOSITORY_URL:
        raise RuntimeError(f"Unexpected existing checkout remote: {observed}")

    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "diff", "--quiet"], check=True
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "fetch", "origin", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "merge", "--ff-only", "FETCH_HEAD"],
        check=True,
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "sync", "--extra", "train", "--extra", "colab"],
    cwd=REPOSITORY_ROOT, check=True,
)
CHECKOUT_REVISION = subprocess.run(
    ["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
PROTOCOL_ROOT = REPOSITORY_ROOT / "notebooks/pathmem_g1_p0/protocol_snapshot"
G0_ROOT = PROTOCOL_ROOT / "experiments/g0"
print({"checkout_revision": CHECKOUT_REVISION,
       "execution_requested": RUN_G1 or RUN_P0})


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/pathmem_g1_p0")
else:
    OUTPUT_BASE = Path("/content/pathmem_g1_p0")

RUN_ROOT = OUTPUT_BASE / RUN_LABEL
G1_ROOT = RUN_ROOT / "g1"
P0_ROOT = RUN_ROOT / "p0"
G1_AUTHORIZATION = G1_ROOT / "g1_authorization.json"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
responsibility_path = RUN_ROOT / "responsible_party.json"
responsibility = {
    "approver": APPROVER.strip(),
    "checkout_revision": CHECKOUT_REVISION,
    "run_label": RUN_LABEL,
    "g1_recipe": G1_RECIPE,
}
if not (RUN_G1 or RUN_P0):
    print("Inspection does not create an execution identity.")
elif responsibility_path.exists():
    if json.loads(responsibility_path.read_text()) != responsibility:
        raise RuntimeError("Run identity changed; choose a new RUN_LABEL")
else:
    temporary = responsibility_path.with_suffix(".json.tmp")
    payload = json.dumps(responsibility, indent=2, sort_keys=True) + "\n"
    temporary.write_text(payload)
    temporary.replace(responsibility_path)

BASE_COMMAND = [
    "uv", "run", "python", "-m", "plasticity_placement.pathmem_exec.cli"
]
plan_command = BASE_COMMAND + [
    "show-plan", "--g0", str(G0_ROOT), "--protocol-root", str(PROTOCOL_ROOT)
]
subprocess.run(plan_command, cwd=REPOSITORY_ROOT, check=True)


In [ ]:
if RUN_G1:
    command = BASE_COMMAND + [
        "g1", "--g0", str(G0_ROOT), "--protocol-root", str(PROTOCOL_ROOT),
        "--output", str(G1_ROOT), "--recipe", G1_RECIPE,
    ]
    subprocess.run(command, cwd=REPOSITORY_ROOT, check=True)
elif RUN_P0:
    if not G1_AUTHORIZATION.is_file():
        raise PermissionError("P0 is blocked: no passing G1 authorization artifact")
    command = BASE_COMMAND + [
        "p0", "--g0", str(G0_ROOT), "--protocol-root", str(PROTOCOL_ROOT),
        "--g1-authorization", str(G1_AUTHORIZATION), "--output", str(P0_ROOT),
    ]
    subprocess.run(command, cwd=REPOSITORY_ROOT, check=True)
else:
    print("Inspection complete: no training or GPU inference was requested.")


In [ ]:
from IPython.display import JSON, display

summaries = (("G1", G1_ROOT / "summary.json"), ("P0", P0_ROOT / "summary.json"))
if (G1_ROOT / "summary.json").is_file():
    diagnostic_path = G1_ROOT / "g1_diagnostic.json"
    subprocess.run(
        BASE_COMMAND + ["diagnose-g1", "--run-root", str(G1_ROOT),
                        "--output", str(diagnostic_path)],
        cwd=REPOSITORY_ROOT, check=True,
    )
for label, path in summaries:
    if path.is_file():
        print(f"{label} summary: {path}")
        display(JSON(json.loads(path.read_text())))
if (G1_ROOT / "g1_diagnostic.json").is_file():
    print(f"G1 diagnostic: {G1_ROOT / 'g1_diagnostic.json'}")
    display(JSON(json.loads((G1_ROOT / "g1_diagnostic.json").read_text())))
print(f"Artifacts root: {RUN_ROOT}")
